# DGA Detection: Autoencoder
The Autoencoder approach treats DGA detection as an unsupervised anomaly detection task. 
It is trained only on "normal" (benign) data. When it encounters a DGA domain, it will fail to reconstruct it accurately, 
resulting in a high "reconstruction error" (the anomaly score). This makes it much harder for new, unseen malware to by-pass the system.

## 1. Data Preparation

Filter the dataset so the model only sees benign domains during the training phase.

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import string
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

In [41]:
# Configuration and hyperparameters
MAX_SEQ_LEN = 75  
BATCH_SIZE = 128
LATENCY_REPS = 100
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Vocab
VOCAB = string.ascii_lowercase + string.digits + "-."
stoi = {ch: i + 1 for i, ch in enumerate(VOCAB)} 
VOCAB_SIZE = len(stoi) + 1

# Encodes domain string into a fixed-length integer sequence
def encode_domain(domain, seq_len=MAX_SEQ_LEN):
    encoded = [stoi.get(c, 0) for c in str(domain).lower()]
    encoded = encoded[:seq_len]
    encoded += [0] * (seq_len - len(encoded))
    return encoded

# Load data
df = pd.read_csv("/Users/maryam/Downloads/dga_data.csv").dropna(subset=['domain'])
df['encoded'] = df['domain'].apply(encode_domain)

# Split into Benign (for training) and Full (for evaluation)
X_all = np.array(df['encoded'].tolist())
y_all = (df['isDGA'] == 'dga').astype(int).values

# Filter training set
X_train_benign = X_all[y_all == 0]
X_train, X_val = train_test_split(X_train_benign, test_size=0.1, random_state=42)

# Convert to PyTorch Tensors
train_tensor = torch.tensor(X_train, dtype=torch.long)
val_tensor = torch.tensor(X_val, dtype=torch.long)
test_full_tensor = torch.tensor(X_all, dtype=torch.long)

## 2. Autoencoder Model

Autoencoder for DGA detection.
Aim of the model is to compresses a domain string into a latent space and attempts to reconstruct it. 

The model consists of:

- Embedding layer
- Linear Encoder to compress the sequence
- Decoder to reconstruct the character logits.

In [44]:
class DGAAutoencoder(nn.Module):

    def __init__(self, vocab_size, embed_dim=32, hidden_dim=128, latent_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(embed_dim * MAX_SEQ_LEN, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim * MAX_SEQ_LEN)
        )
        
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        batch_size = x.size(0)
        # Embedding
        emb = self.embedding(x)
        
        # Encode
        latent = self.encoder(emb)
        
        # Decode
        reconstructed_flat = self.decoder(latent)
        reconstructed_emb = reconstructed_flat.view(batch_size, MAX_SEQ_LEN, -1)
        
        # Project to Logits
        logits = self.output_layer(reconstructed_emb)
        return logits

# Initialize Model
ae_model = DGAAutoencoder(VOCAB_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(reduction='none')

## 3. Training

Training on benign data focuses on minimizing the reconstruction loss for legitimate domain strings.

In [47]:
train_loader = DataLoader(train_tensor, batch_size=BATCH_SIZE, shuffle=True)

ae_model.train()
for epoch in range(15):
    epoch_loss = 0
    for batch in train_loader:
        x = batch.to(DEVICE)
        optimizer.zero_grad()
        
        logits = ae_model(x)

        loss = criterion(logits.transpose(1, 2), x).mean()
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} : Avg Loss: {epoch_loss/len(train_loader):.4f}")

Epoch 05 : Avg Loss: 0.0070
Epoch 10 : Avg Loss: 0.0067
Epoch 15 : Avg Loss: 0.0059


## 4. Anomaly Detection & Inference Latency
Calculate the reconstruction error for the entire test set (benign + DGA) to find anomaly score.

In [49]:
ae_model.eval()
test_loader = DataLoader(test_full_tensor, batch_size=256, shuffle=False)
anomaly_scores = []

# Latency
start_time = time.perf_counter()
with torch.no_grad():
    for batch in test_loader:
        x = batch.to(DEVICE)
        logits = ae_model(x)

        char_losses = criterion(logits.transpose(1, 2), x)
        
        #Anomaly Score
        sample_losses = char_losses.mean(dim=1)
        anomaly_scores.extend(sample_losses.cpu().numpy())

total_time = (time.perf_counter() - start_time)
latency_ms = (total_time / len(X_all)) * 1000

# Performance Metrics
auc_roc = roc_auc_score(y_all, anomaly_scores)
pr_auc = average_precision_score(y_all, anomaly_scores)

print(f"\n--- Autoencoder Results ---")
print(f"AUC-ROC:           {auc_roc:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")
print(f"Inference Latency: {latency_ms:.4f} ms/sample")
print(f"Parameter Count:   {sum(p.numel() for p in ae_model.parameters()):,}")


--- Autoencoder Results ---
AUC-ROC:           0.8903
PR-AUC:            0.9117
Inference Latency: 0.0092 ms/sample
Parameter Count:   627,815
